# Multigenome numerical statistics
Read the prepared HDF5 in chunks; count annotation statuses and the number of targets containing a repeat with the same name. `not_compared` is missing comparison data, not evidence of absence. No tree, ancestral-state reconstruction, figures or GC analysis. The original status TSV files remain the authoritative complete classification output.

In [ ]:
from pathlib import Path
import json
import sys
import csv
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "bin" / "annotation_io.py").is_file():
    raise RuntimeError("Start Jupyter from the repository or its notebooks directory")
sys.path.insert(0, str(ROOT / "bin"))
from annotation_io import iter_bed

In [ ]:
WORK_H5 = ROOT / "results/analysis/comparison.h5"
OUTPUT_DIR = ROOT / "results/analysis"
CHUNKSIZE = 250_000

In [ ]:
META_COLUMNS = ["source_tag", "source_genome_id", "group_id", "ref_chrom", "ref_start", "ref_end", "repeat_len", "repeat_ids_inside"]
RETAINED = {"transferred_one_repeat_same_name", "transferred_multiple_repeats_with_source_name"}
annotation_counts = Counter()
retained_counts = Counter()
with pd.HDFStore(WORK_H5, mode="r") as store:
    tags = list(store.get_storer("repeat_annotation_status_matrix").attrs.genome_tags)
    for chunk in store.select("repeat_annotation_status_matrix", iterator=True, chunksize=CHUNKSIZE):
        for source, frame in chunk.groupby("source_tag", sort=False):
            targets = [tag for tag in tags if tag != source]
            for target in targets:
                for status, count in frame[target].value_counts(dropna=False).items():
                    annotation_counts[(source,target,status)] += int(count)
            n_retained = frame[targets].isin(RETAINED).sum(axis=1)
            n_compared = (~frame[targets].isin(["not_compared"])).sum(axis=1)
            for key, count in pd.DataFrame({"retained":n_retained,"compared":n_compared}).value_counts().items():
                retained_counts[(source,int(key[0]),int(key[1]))] += int(count)
annotation_summary = pd.DataFrame([(*key,count) for key,count in annotation_counts.items()],
                                  columns=["source","target","annotation_status","count"])
retention_summary = pd.DataFrame([(*key,count) for key,count in retained_counts.items()],
                                 columns=["source","n_retained_targets","n_compared_targets","count"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
annotation_summary.to_csv(OUTPUT_DIR / "annotation_status_counts.tsv", sep="\t", index=False)
retention_summary.to_csv(OUTPUT_DIR / "retained_target_counts.tsv", sep="\t", index=False)
annotation_summary